# Projet Challenge 2 – Modules 1 et 2 : exploration et variables

Données : échantillon IBM TabFormer (500 utilisateurs, décembre 2014 → 2019) sauvegardé dans le Drive.

- **Module 1** : taux de fraude, montants par classe, évolution dans le temps, fraude par heure, catégorie et canal.
- **Module 2** : nettoyage, variables temporelles, comportementales (sans fuite) et contextuelles, puis sauvegarde.

Règle d'or : **chaque variable d'une transaction n'utilise que le passé de sa carte.**

## 0. Chargement de l'échantillon

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DOSSIER = "/content/drive/MyDrive/fraude_tabformer"
ech = pd.read_parquet(f"{DOSSIER}/echantillon_500_users.parquet")

print(f"{len(ech):,} transactions | {ech['User'].nunique()} utilisateurs | "
      f"{ech.groupby(['User', 'Card']).ngroups:,} cartes")
print(ech["partie"].value_counts().to_string())

---
# Module 1 – Exploration

On explore **2015–2019** (sans la période de chauffe). Pour rester honnête, les conclusions qui guident des choix
de modélisation se vérifient sur l'**entraînement** ; ici on regarde l'ensemble pour comprendre les données.

In [ ]:
eda = ech[ech["partie"] != "chauffe"]
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})

print(f"Taux de fraude global : {eda['fraude'].mean() * 100:.3f} %  ({eda['fraude'].sum():,} fraudes)")
print(f"Montant total fraudé : {eda.loc[eda['fraude'] == 1, 'Amount'].sum():,.0f} $")
eda.groupby("partie")["fraude"].agg(transactions="size", fraudes="sum", taux="mean")

### 1.1 Montants par classe

In [ ]:
print(eda.groupby("fraude")["Amount"].describe(percentiles=[.25, .5, .75, .95, .99]).round(2).to_string())
print(f"\nMontants négatifs (remboursements) : {(eda['Amount'] < 0).sum():,} "
      f"dont fraudes : {((eda['Amount'] < 0) & (eda['fraude'] == 1)).sum():,}")

fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
pos = eda[eda["Amount"] > 0]
for c, lab, col in [(0, "normale", "steelblue"), (1, "fraude", "crimson")]:
    np.log10(pos.loc[pos["fraude"] == c, "Amount"]).plot.hist(
        bins=80, density=True, alpha=0.55, ax=ax[0], label=lab, color=col)
ax[0].set_title("Distribution des montants (échelle log10)"); ax[0].set_xlabel("log10(montant $)"); ax[0].legend()

tranches = pd.cut(eda["Amount"], [-np.inf, 0, 10, 50, 100, 250, 500, 1000, np.inf],
                  labels=["≤0", "0-10", "10-50", "50-100", "100-250", "250-500", "500-1000", ">1000"])
(eda.groupby(tranches, observed=True)["fraude"].mean() * 100).plot.bar(ax=ax[1], color="crimson")
ax[1].set_title("Taux de fraude par tranche de montant (%)"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()

### 1.2 Évolution du taux de fraude dans le temps

In [ ]:
mois = eda.groupby(eda["date"].dt.to_period("M")).agg(transactions=("fraude", "size"), fraudes=("fraude", "sum"))
mois["taux_%"] = mois["fraudes"] / mois["transactions"] * 100

fig, ax = plt.subplots(figsize=(13, 3.5))
mois["taux_%"].plot(ax=ax, color="crimson", marker=".")
for debut, nom in [("2018-01", "validation"), ("2019-01", "test")]:
    ax.axvline(pd.Period(debut, "M"), color="grey", ls="--"); ax.text(pd.Period(debut, "M"), ax.get_ylim()[1] * 0.9, f" {nom}")
ax.set_title("Taux de fraude mensuel (%)"); ax.set_xlabel("")
plt.tight_layout(); plt.show()
print(f"Mois sans aucune fraude : {(mois['fraudes'] == 0).sum()} sur {len(mois)}")

### 1.3 Fraude par heure et par jour de la semaine

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 3.5))
(eda.groupby(eda["date"].dt.hour)["fraude"].mean() * 100).plot.bar(ax=ax[0], color="crimson")
ax[0].set_title("Taux de fraude par heure (%)"); ax[0].set_xlabel("heure")
jours = ["lun", "mar", "mer", "jeu", "ven", "sam", "dim"]
t = eda.groupby(eda["date"].dt.dayofweek)["fraude"].mean() * 100
t.index = jours; t.plot.bar(ax=ax[1], color="crimson")
ax[1].set_title("Taux de fraude par jour (%)"); ax[1].set_xlabel("")
plt.tight_layout(); plt.show()

### 1.4 Fraude par canal, erreur, catégorie de marchand et lieu

In [ ]:
def taux_par(serie, n=12, min_tx=200):
    t = eda.groupby(serie, observed=True)["fraude"].agg(transactions="size", fraudes="sum", taux="mean")
    t = t[t["transactions"] >= min_tx]
    t["taux"] = (t["taux"] * 100).round(3)
    return t.sort_values("taux", ascending=False).head(n)

print("--- Canal ---");  print(taux_par(eda["Use Chip"]).to_string())
erreur = eda["Errors?"].astype(str).replace({"nan": "aucune", "None": "aucune"})
print("\n--- Erreurs (≥ 200 transactions) ---"); print(taux_par(erreur).to_string())
print("\n--- MCC (≥ 200 transactions) ---");     print(taux_par(eda["MCC"]).to_string())

In [ ]:
# Lieu : en ligne, aux États-Unis, ou à l'étranger (pour l'étranger, "Merchant State" contient le nom du pays)
etat = eda["Merchant State"].astype(str)
lieu = np.select([eda["Use Chip"].astype(str).str.contains("Online"), etat.str.len() > 2],
                 ["en ligne", "étranger"], default="États-Unis")
print(taux_par(pd.Series(lieu, index=eda.index, name="lieu")).to_string())

### 1.5 Concentration des fraudes par marchand

Dans un jeu synthétique, une catégorie à fort taux peut venir de **quelques marchands** seulement.
Si c'est le cas, le modèle risque d'apprendre « ce marchand » au lieu d'un comportement.

In [ ]:
fr = eda[eda["fraude"] == 1]
par_marchand = fr["Merchant Name"].value_counts()
print(f"{fr['Merchant Name'].nunique():,} marchands distincts pour {len(fr):,} fraudes")
print(f"Les 10 marchands les plus fraudés concentrent {par_marchand.head(10).sum() / len(fr) * 100:.1f} % des fraudes")

# Les marchands fraudés en test l'étaient-ils déjà pendant l'entraînement ?
m_train = set(fr.loc[fr["partie"] == "entraînement", "Merchant Name"])
for p in ["validation", "test"]:
    m = fr.loc[fr["partie"] == p, "Merchant Name"]
    print(f"Fraudes {p} chez un marchand déjà fraudé en entraînement : {m.isin(m_train).mean() * 100:.1f} %")

### 1.6 Les fraudes arrivent-elles en rafale ?

In [ ]:
fr_tri = fr.sort_values(["User", "Card", "date"])
ecart = fr_tri.groupby(["User", "Card"])["date"].diff().dt.total_seconds() / 3600
print("Temps entre deux fraudes successives d'une même carte (heures) :")
print(ecart.describe(percentiles=[.25, .5, .75, .9]).round(1).to_string())
episodes = fr_tri.groupby(["User", "Card"]).size()
print(f"\nFraudes par carte fraudée : médiane {episodes.median():.0f}, max {episodes.max()}")

---
# Module 2 – Nettoyage et construction des variables

Les variables sont calculées sur **tout l'historique trié** (y compris la chauffe de décembre 2014),
en ne regardant que le **passé** de chaque carte. Elles ne « voient » donc jamais la cible ni le futur :
on peut les calculer avant la séparation sans fuite.

### 2.1 Nettoyage

In [ ]:
data = ech.copy()
data["carte"] = data["User"].astype(str) + "_" + data["Card"].astype(str)   # identifiant de carte (groupement seulement)

# Tri : indispensable pour les fenêtres temporelles
data = data.sort_values(["carte", "date"], kind="mergesort").reset_index(drop=True)

print(f"Doublons exacts : {data.duplicated().sum():,}")
print(f"Montants nuls : {(data['Amount'] == 0).sum():,} | négatifs : {(data['Amount'] < 0).sum():,}")

### 2.2 Variables temporelles

In [ ]:
h = data["date"].dt.hour + data["date"].dt.minute / 60
data["heure_sin"] = np.sin(2 * np.pi * h / 24)
data["heure_cos"] = np.cos(2 * np.pi * h / 24)
data["jour_semaine"] = data["date"].dt.dayofweek.astype("int8")
data["week_end"] = (data["jour_semaine"] >= 5).astype("int8")
data["nuit"] = ((h >= 0) & (h < 6)).astype("int8")          # 0 h – 6 h : lisible pour les codes motifs

### 2.3 Variables de transaction : montant, canal, lieu, erreurs, catégorie

In [ ]:
data["montant"] = data["Amount"].astype("float32")
data["remboursement"] = (data["montant"] < 0).astype("int8")
data["log_montant"] = np.sign(data["montant"]) * np.log1p(data["montant"].abs())

canal = data["Use Chip"].astype(str)
data["en_ligne"] = canal.str.contains("Online").astype("int8")
data["bande"] = canal.str.contains("Swipe").astype("int8")       # puce = référence (ni en ligne ni bande)

etat = data["Merchant State"].astype(str)
data["etranger"] = ((etat.str.len() > 2) & (data["en_ligne"] == 0)).astype("int8")

err = data["Errors?"].astype(str)
for nom, motif in {"err_cvv": "Bad CVV", "err_expiration": "Bad Expiration", "err_numero": "Bad Card Number",
                   "err_pin": "Bad PIN", "err_solde": "Insufficient Balance", "err_technique": "Technical Glitch",
                   "err_code_postal": "Bad Zipcode"}.items():
    data[nom] = err.str.contains(motif, regex=False).astype("int8")
data["une_erreur"] = data.filter(like="err_").max(axis=1).astype("int8")

# MCC : regroupement des codes de voyage (compagnies aériennes, location de voitures, hôtels)
mcc = data["MCC"].astype(int)
data["mcc_groupe"] = np.select(
    [mcc.between(3000, 3299), mcc.between(3351, 3441), mcc.between(3501, 3999)],
    ["aerien", "location_voiture", "hotel"], default=mcc.astype(str))
print("Erreurs :", data.filter(like="err_").sum().to_dict())
print(f"Groupes MCC : {data['mcc_groupe'].nunique()}")

### 2.4 Variables comportementales par carte (uniquement le passé)

`closed="left"` exclut la transaction en cours de chaque fenêtre : c'est ce qui évite la fuite.

In [ ]:
cle = ["carte"]
g = data.groupby(cle, sort=False)

# Temps depuis la transaction précédente de la carte (heures)
data["h_depuis_prec"] = (g["date"].diff().dt.total_seconds() / 3600).astype("float32")

# Fenêtres glissantes sur le passé, par carte
idx = data.set_index("date")
roul = idx.groupby("carte", sort=False)

def fenetre(col, taille, fonction):
    r = getattr(roul[col].rolling(taille, closed="left"), fonction)()
    return r.to_numpy()          # même ordre que data (trié par carte puis date)

data["nb_tx_1h"]   = np.nan_to_num(fenetre("montant", "1h",  "count")).astype("int16")
data["nb_tx_24h"]  = np.nan_to_num(fenetre("montant", "24h", "count")).astype("int16")
data["nb_err_24h"] = np.nan_to_num(fenetre("une_erreur", "24h", "sum")).astype("int16")
data["moy_30j"]    = fenetre("montant", "30D", "mean").astype("float32")
data["nb_tx_30j"]  = np.nan_to_num(fenetre("montant", "30D", "count")).astype("int16")

# Montant rapporté à la moyenne des 30 derniers jours (1 = habituel ; NaN si aucun historique)
data["ratio_montant_30j"] = (data["montant"] / (data["moy_30j"].abs() + 1)).astype("float32")

# Nouveauté : premier achat de la carte chez ce marchand / dans ce groupe MCC / dans cette ville
data["nouveau_marchand"] = (data.groupby(["carte", "Merchant Name"]).cumcount() == 0).astype("int8")
data["nouvelle_categorie"] = (data.groupby(["carte", "mcc_groupe"]).cumcount() == 0).astype("int8")
ville = data["Merchant City"].astype(str)
data["ville_differente_prec"] = ((ville != g["Merchant City"].shift().astype(str))
                                 & (data["en_ligne"] == 0)).astype("int8")

# Part des transactions en ligne de la carte jusqu'ici (habitude)
data["part_en_ligne_passee"] = (g["en_ligne"].cumsum() - data["en_ligne"]) / g.cumcount().replace(0, np.nan)

data.filter(regex="^(nb_|ratio|h_depuis|nouveau|nouvelle|ville_diff|part_)").describe().T.round(3)

### 2.5 Contrôle anti-fuite

Test simple : si on **supprime** une transaction, ses propres variables comportementales ne doivent pas changer,
et les variables des transactions **antérieures** non plus. On le vérifie sur une carte au hasard.

In [ ]:
def comportement(df_carte):
    d = df_carte.sort_values("date").set_index("date")
    return pd.DataFrame({
        "nb_tx_1h": d["montant"].rolling("1h", closed="left").count().fillna(0).to_numpy(),
        "nb_tx_24h": d["montant"].rolling("24h", closed="left").count().fillna(0).to_numpy(),
        "moy_30j": d["montant"].rolling("30D", closed="left").mean().to_numpy(),
    }, index=df_carte.sort_values("date").index)

carte_test = data.loc[data["fraude"] == 1, "carte"].iloc[0]
c = data[data["carte"] == carte_test]
ref = comportement(c)
i = len(c) // 2
apres_suppr = comportement(c.drop(c.index[i + 1:]))      # on retire tout le futur
ecart = (ref.iloc[: i + 1] - apres_suppr).abs().max()
print("Écart max quand on retire le futur (doit être 0) :")
print(ecart.round(6).to_string())

# Cohérence avec les colonnes calculées en bloc
bloc = data.loc[c.index, ["nb_tx_1h", "nb_tx_24h", "moy_30j"]]
print("\nÉcart avec le calcul en bloc (doit être 0) :")
print((bloc.reset_index(drop=True) - ref.reset_index(drop=True)).abs().max().round(6).to_string())

### 2.6 Premier regard sur le signal (entraînement seulement)

In [ ]:
tr = data[data["partie"] == "entraînement"]
def signal(col, bins=None):
    s = pd.cut(tr[col], bins) if bins is not None else tr[col]
    t = tr.groupby(s, observed=True)["fraude"].agg(transactions="size", taux="mean")
    t["taux"] = (t["taux"] * 100).round(3)
    return t

print("Nouveau marchand :\n", signal("nouveau_marchand").to_string())
print("\nNombre de transactions sur 1 h :\n", signal("nb_tx_1h", [-1, 0, 1, 2, 5, np.inf]).to_string())
print("\nErreurs sur 24 h :\n", signal("nb_err_24h", [-1, 0, 1, 3, np.inf]).to_string())
print("\nMontant / moyenne 30 j :\n", signal("ratio_montant_30j", [-np.inf, 0, 0.5, 1, 2, 5, 10, np.inf]).to_string())
print("\nVille différente de la précédente :\n", signal("ville_differente_prec").to_string())

### 2.7 Sélection des colonnes et sauvegarde

In [ ]:
VARIABLES = [
    # temporelles
    "heure_sin", "heure_cos", "jour_semaine", "week_end", "nuit",
    # transaction
    "montant", "log_montant", "remboursement", "en_ligne", "bande", "etranger",
    "err_cvv", "err_expiration", "err_numero", "err_pin", "err_solde", "err_technique", "err_code_postal", "une_erreur",
    "mcc_groupe",
    # comportementales (passé de la carte)
    "h_depuis_prec", "nb_tx_1h", "nb_tx_24h", "nb_err_24h", "nb_tx_30j", "moy_30j", "ratio_montant_30j",
    "nouveau_marchand", "nouvelle_categorie", "ville_differente_prec", "part_en_ligne_passee",
]
GARDER = ["date", "carte", "partie", "fraude"] + VARIABLES   # carte : pour l'analyse, jamais donnée au modèle

final = data.loc[data["partie"] != "chauffe", GARDER].sort_values("date").reset_index(drop=True)
final.to_parquet(f"{DOSSIER}/variables_v1.parquet", index=False)

print(f"{len(final):,} lignes x {len(VARIABLES)} variables -> variables_v1.parquet")
print(final.groupby("partie")["fraude"].agg(transactions="size", fraudes="sum").to_string())
print("\nValeurs manquantes :", final[VARIABLES].isna().sum()[lambda s: s > 0].to_dict())

## À rapporter pour l'analyse

Copie les sorties texte (pas les tableaux HTML) de : **1.1, 1.2 (mois sans fraude), 1.4, 1.5, 1.6, 2.5 et 2.6**,
plus les graphiques 1.1 à 1.3 en capture si tu veux.